# 双均线策略回测

本 Notebook 实现以下功能：
1. 读取股票数据
2. 提取市值在 100 亿以上的主板票
3. 时间序列数据集划分（训练/验证/测试），严格无穿越
4. 双均线策略回测

## 1. 环境配置与导入

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# 添加项目路径
project_path = os.path.dirname(os.path.abspath('__file__'))
if project_path not in sys.path:
    sys.path.insert(0, project_path)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# 量化库
import backtrader as bt
import vectorbt as vbt

# 项目工具
from utils.data_processor import DataProcessor
from utils.factor_calculator import TechnicalFactorCalculator
from utils.logger import get_logger

logger = get_logger(__name__)
print("✅ 环境配置完成")

## 2. 数据获取与加载

支持两种数据源：
- **本地 CSV 文件**: 如果你有历史数据文件
- **在线 API**: 使用 AKShare/Tushare 实时获取

In [ ]:
class StockDataLoader:
    """
    股票数据加载器
    支持本地 CSV 或在线 API
    """
    
    def __init__(self, data_path=None):
        self.data_path = data_path
        self.raw_df = None
        
    def load_from_csv(self, filepath, nrows=None):
        """从 CSV 加载数据"""
        print(f"📊 加载数据: {filepath}")
        
        df = pd.read_csv(filepath, nrows=nrows)
        
        # 标准字段映射
        column_mapping = {
            'stock_code': 'code',
            'ts_code': 'code',
            'trade_date': 'date',
            'close_price': 'close',
            'open_price': 'open',
            'high_price': 'high',
            'low_price': 'low',
            'vol': 'volume',
            'amount': 'amount',
            'market_cap': 'market_cap',
            'total_mv': 'market_cap',  # Tushare 市值字段
            'circ_mv': 'circ_market_cap'
        }
        
        # 重命名列
        df = df.rename(columns={k: v for k, v in column_mapping.items() if k in df.columns})
        
        # 转换日期
        if 'date' in df.columns:
            df['date'] = pd.to_datetime(df['date'])
        
        # 标准化股票代码（6位数字）
        if 'code' in df.columns:
            df['code'] = df['code'].astype(str).str.zfill(6)
            # 提取交易所信息
            df['exchange'] = df['code'].apply(self._get_exchange)
        
        self.raw_df = df
        print(f"✅ 加载完成: {len(df):,} 条记录, {df['code'].nunique()} 只股票")
        return df
    
    def _get_exchange(self, code):
        """根据代码判断交易所"""
        if code.startswith('6'):
            return 'SH'  # 上海主板
        elif code.startswith('0') or code.startswith('001'):
            return 'SZ'  # 深圳主板
        elif code.startswith('3'):
            return 'CY'  # 创业板
        elif code.startswith('68'):
            return 'KC'  # 科创板
        elif code.startswith('8') or code.startswith('4'):
            return 'BJ'  # 北交所
        return 'OTHER'
    
    def load_from_api(self, stock_codes, start_date, end_date):
        """从 AKShare 获取数据"""
        try:
            import akshare as ak
            
            all_data = []
            for code in stock_codes:
                print(f"📥 获取 {code}...")
                df = ak.stock_zh_a_hist(symbol=code, period="daily", 
                                       start_date=start_date, end_date=end_date, adjust="qfq")
                df['code'] = code
                all_data.append(df)
            
            df = pd.concat(all_data, ignore_index=True)
            df = df.rename(columns={
                '日期': 'date',
                '开盘': 'open',
                '收盘': 'close',
                '最高': 'high',
                '最低': 'low',
                '成交量': 'volume',
                '成交额': 'amount'
            })
            df['date'] = pd.to_datetime(df['date'])
            df['exchange'] = df['code'].apply(self._get_exchange)
            
            self.raw_df = df
            return df
        except Exception as e:
            print(f"❌ 获取数据失败: {e}")
            return None

# 实例化加载器
loader = StockDataLoader()
print("✅ 数据加载器初始化完成")

### 2.1 加载数据（根据实际情况选择）

**选项 A**: 如果你有本地 CSV 文件

In [ ]:
# 选项 A: 从本地 CSV 加载（如果有的话）
# csv_path = "data/a_stock_history_price.csv"  # 修改为你的文件路径
# df_raw = loader.load_from_csv(csv_path)

# 选项 B: 从 API 获取示例数据（这里用几只大盘股作为示例）
sample_codes = ['000001', '000002', '600519', '601318', '000858']  # 平安银行、万科、茅台、平安、五粮液

# 获取数据
end_date = datetime.now().strftime('%Y%m%d')
start_date = (datetime.now() - timedelta(days=365*3)).strftime('%Y%m%d')

df_raw = loader.load_from_api(sample_codes, start_date, end_date)

print(f"\n📊 数据概览:")
print(df_raw.head())
print(f"\n股票列表: {df_raw['code'].unique()}")

### 2.2 添加市值数据（模拟/真实）

如果原始数据中没有市值，我们需要补充

In [ ]:
def add_market_cap(df, is_mock=True):
    """
    添加市值数据
    如果 is_mock=True，使用模拟的市值数据
    """
    if is_mock or 'market_cap' not in df.columns:
        print("⚠️ 使用模拟市值数据（仅用于演示）")
        
        # 模拟市值数据（单位：亿元）
        mock_cap = {
            '000001': 2500,   # 平安银行 ~2500亿
            '000002': 1500,   # 万科 ~1500亿
            '600519': 21000,  # 茅台 ~2.1万亿
            '601318': 8500,   # 中国平安 ~8500亿
            '000858': 6500,   # 五粮液 ~6500亿
        }
        
        df['market_cap'] = df['code'].map(mock_cap)
        
        # 为没有映射的股票添加默认值
        df['market_cap'] = df['market_cap'].fillna(200)  # 默认200亿
    
    # 确保市值单位为亿元
    if df['market_cap'].max() > 1e8:
        df['market_cap'] = df['market_cap'] / 1e8  # 转为亿元
    
    return df

# 添加市值数据
df_raw = add_market_cap(df_raw, is_mock=True)

# 查看市值分布
print("\n📊 市值分布（亿元）:")
cap_summary = df_raw.groupby('code')['market_cap'].first().sort_values(ascending=False)
print(cap_summary)

## 3. 数据筛选：100亿以上主板票

In [ ]:
def filter_mainboard_largecap(df, min_cap=100):
    """
    筛选主板大市值股票
    
    Parameters:
    -----------
    df : DataFrame
        原始数据
    min_cap : float
        最小市值（亿元），默认100亿
    """
    print(f"\n🔍 筛选条件: 主板 + 市值 >= {min_cap}亿")
    
    # 筛选主板（上海主板 SH，深圳主板 SZ）
    mainboard = df[df['exchange'].isin(['SH', 'SZ'])]
    print(f"   主板股票: {mainboard['code'].nunique()} 只")
    
    # 筛选大市值
    largecap = mainboard[mainboard['market_cap'] >= min_cap]
    print(f"   市值>={min_cap}亿: {largecap['code'].nunique()} 只")
    
    # 显示筛选结果
    selected_codes = largecap['code'].unique()
    print(f"\n✅ 筛选后股票 ({len(selected_codes)} 只):")
    for code in selected_codes:
        cap = largecap[largecap['code']==code]['market_cap'].iloc[0]
        exch = largecap[largecap['code']==code]['exchange'].iloc[0]
        print(f"   {code} ({exch}): {cap:.0f}亿")
    
    return largecap

# 执行筛选
df_filtered = filter_mainboard_largecap(df_raw, min_cap=100)

# 按股票代码分组查看时间范围
print("\n📅 各股票数据时间范围:")
for code in df_filtered['code'].unique():
    stock_df = df_filtered[df_filtered['code'] == code]
    print(f"   {code}: {stock_df['date'].min().date()} ~ {stock_df['date'].max().date()} ({len(stock_df)} 天)")

## 4. 时间序列数据划分（无穿越）

**关键原则**: 按时间划分，确保训练集的时间早于验证集，验证集早于测试集

```
时间轴: |---- 训练集 ----|---- 验证集 ----|---- 测试集 ---->|
        t0             t1             t2              t3
```

In [ ]:
class TimeSeriesSplitter:
    """
    时间序列数据集划分器
    确保严格无穿越（训练 -> 验证 -> 测试）
    """
    
    def __init__(self, df, date_col='date'):
        self.df = df.copy()
        self.date_col = date_col
        self.train_df = None
        self.val_df = None
        self.test_df = None
    
    def split_by_ratio(self, train_ratio=0.6, val_ratio=0.2, test_ratio=0.2):
        """
        按比例划分数据集
        
        Parameters:
        -----------
        train_ratio : 训练集比例
        val_ratio : 验证集比例
        test_ratio : 测试集比例
        """
        assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6, "比例之和必须等于1"
        
        # 获取全局日期范围
        min_date = self.df[self.date_col].min()
        max_date = self.df[self.date_col].max()
        total_days = (max_date - min_date).days
        
        # 计算分割点
        train_end = min_date + timedelta(days=int(total_days * train_ratio))
        val_end = train_end + timedelta(days=int(total_days * val_ratio))
        
        print(f"📊 数据集划分:")
        print(f"   总时间范围: {min_date.date()} ~ {max_date.date()} ({total_days} 天)")
        print(f"   训练集: {min_date.date()} ~ {train_end.date()}")
        print(f"   验证集: {train_end.date()} ~ {val_end.date()}")
        print(f"   测试集: {val_end.date()} ~ {max_date.date()}")
        
        # 划分数据
        self.train_df = self.df[self.df[self.date_col] < train_end].copy()
        self.val_df = self.df[(self.df[self.date_col] >= train_end) & 
                              (self.df[self.date_col] < val_end)].copy()
        self.test_df = self.df[self.df[self.date_col] >= val_end].copy()
        
        # 统计各集合
        self._print_stats()
        
        return self.train_df, self.val_df, self.test_df
    
    def split_by_date(self, train_end, val_end):
        """按指定日期划分"""
        train_end = pd.to_datetime(train_end)
        val_end = pd.to_datetime(val_end)
        
        self.train_df = self.df[self.df[self.date_col] < train_end].copy()
        self.val_df = self.df[(self.df[self.date_col] >= train_end) & 
                              (self.df[self.date_col] < val_end)].copy()
        self.test_df = self.df[self.df[self.date_col] >= val_end].copy()
        
        self._print_stats()
        return self.train_df, self.val_df, self.test_df
    
    def _print_stats(self):
        """打印统计信息"""
        for name, df in [('训练集', self.train_df), ('验证集', self.val_df), ('测试集', self.test_df)]:
            if len(df) > 0:
                n_stocks = df['code'].nunique()
                n_records = len(df)
                date_range = f"{df[self.date_col].min().date()} ~ {df[self.date_col].max().date()}"
                print(f"\n   {name}: {n_records:,} 条记录, {n_stocks} 只股票")
                print(f"          时间: {date_range}")

# 执行划分
splitter = TimeSeriesSplitter(df_filtered)
train_df, val_df, test_df = splitter.split_by_ratio(train_ratio=0.6, val_ratio=0.2, test_ratio=0.2)

### 4.1 可视化数据划分

In [ ]:
def plot_data_split(splitter, stock_code=None):
    """可视化数据划分"""
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    
    # 选择一只股票或取平均
    if stock_code:
        train = splitter.train_df[splitter.train_df['code'] == stock_code]
        val = splitter.val_df[splitter.val_df['code'] == stock_code]
        test = splitter.test_df[splitter.test_df['code'] == stock_code]
        title_suffix = f" (股票: {stock_code})"
    else:
        # 取所有股票的平均价格
        train = splitter.train_df.groupby('date')['close'].mean().reset_index()
        val = splitter.val_df.groupby('date')['close'].mean().reset_index()
        test = splitter.test_df.groupby('date')['close'].mean().reset_index()
        title_suffix = " (全市场平均)"
    
    ax1 = axes[0]
    ax1.plot(train['date'], train['close'], label='训练集', color='blue', alpha=0.7)
    ax1.plot(val['date'], val['close'], label='验证集', color='orange', alpha=0.7)
    ax1.plot(test['date'], test['close'], label='测试集', color='green', alpha=0.7)
    ax1.axvline(splitter.val_df['date'].min(), color='red', linestyle='--', alpha=0.5, label='训练/验证分界')
    ax1.axvline(splitter.test_df['date'].min(), color='purple', linestyle='--', alpha=0.5, label='验证/测试分界')
    ax1.set_title(f'价格走势与数据划分{title_suffix}')
    ax1.set_ylabel('价格')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 数据量统计
    ax2 = axes[1]
    sets = ['训练集', '验证集', '测试集']
    counts = [len(splitter.train_df), len(splitter.val_df), len(splitter.test_df)]
    colors = ['blue', 'orange', 'green']
    bars = ax2.bar(sets, counts, color=colors, alpha=0.6)
    ax2.set_title('各数据集样本数')
    ax2.set_ylabel('记录数')
    
    # 添加数值标签
    for bar in bars:
        height = bar.get_height()
        ax2.annotate(f'{int(height):,}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()

# 可视化
plot_data_split(splitter, stock_code='600519')  # 以茅台为例

## 5. 双均线策略实现

**策略逻辑**:
- **买入信号**: 短期均线上穿长期均线（金叉）
- **卖出信号**: 短期均线下穿长期均线（死叉）
- **参数**: 短期均线(5日/10日)，长期均线(20日/60日)

In [ ]:
class DualMAStrategy(bt.Strategy):
    """
    双均线策略
    
    参数:
        short_period (int): 短期均线周期，默认10
        long_period (int): 长期均线周期，默认30
        position_pct (float): 每次开仓资金比例，默认0.95 (95%)
    """
    
    params = (
        ('short_period', 10),     # 短期均线
        ('long_period', 30),      # 长期均线
        ('position_pct', 0.95),   # 资金使用比例
        ('print_log', True),      # 是否打印日志
    )
    
    def __init__(self):
        # 保存订单引用
        self.order = None
        self.buy_price = None
        self.sell_price = None
        
        # 计算均线
        self.ma_short = bt.indicators.SMA(self.data.close, period=self.params.short_period)
        self.ma_long = bt.indicators.SMA(self.data.close, period=self.params.long_period)
        
        # 金叉/死叉信号
        self.crossover = bt.indicators.CrossOver(self.ma_short, self.ma_long)
        
        # 记录交易
        self.trades = []
    
    def log(self, txt, dt=None):
        """日志函数"""
        if self.params.print_log:
            dt = dt or self.datas[0].datetime.date(0)
            print(f'{dt.isoformat()} {txt}')
    
    def notify_order(self, order):
        """订单状态通知"""
        if order.status in [order.Submitted, order.Accepted]:
            return
        
        if order.status in [order.Completed]:
            if order.isbuy():
                self.log(f'【买入】价格: {order.executed.price:.2f}, '
                        f'数量: {order.executed.size}, '
                        f'成本: {order.executed.value:.2f}')
                self.buy_price = order.executed.price
            else:
                self.log(f'【卖出】价格: {order.executed.price:.2f}, '
                        f'数量: {order.executed.size}, '
                        f'收入: {order.executed.value:.2f}')
                
                # 计算盈亏
                if self.buy_price:
                    pnl = order.executed.price - self.buy_price
                    pnl_pct = (order.executed.price / self.buy_price - 1) * 100
                    self.log(f'      盈亏: {pnl:.2f} ({pnl_pct:+.2f}%)')
                    
                    self.trades.append({
                        'buy_price': self.buy_price,
                        'sell_price': order.executed.price,
                        'pnl': pnl,
                        'pnl_pct': pnl_pct
                    })
        
        self.order = None
    
    def notify_trade(self, trade):
        """交易完成通知"""
        if not trade.isclosed:
            return
        self.log(f'【交易完成】毛利润: {trade.pnl:.2f}, 净利润: {trade.pnlcomm:.2f}')
    
    def next(self):
        """每个bar执行一次"""
        # 如果正在下单，不操作
        if self.order:
            return
        
        # 如果没有持仓
        if not self.position:
            # 金叉买入信号
            if self.crossover > 0:
                size = int(self.broker.getcash() * self.params.position_pct / self.data.close[0] / 100) * 100
                if size >= 100:
                    self.log(f'【信号】金叉买入 | 价格: {self.data.close[0]:.2f}, '
                            f'MA{self.params.short_period}: {self.ma_short[0]:.2f}, '
                            f'MA{self.params.long_period}: {self.ma_long[0]:.2f}')
                    self.order = self.buy(size=size)
        
        else:
            # 死叉卖出信号
            if self.crossover < 0:
                self.log(f'【信号】死叉卖出 | 价格: {self.data.close[0]:.2f}, '
                        f'持仓: {self.position.size}')
                self.order = self.sell(size=self.position.size)

print("✅ 双均线策略类定义完成")

### 5.1 回测执行函数

In [ ]:
def run_backtest(df, strategy_class, strategy_params=None, 
                 initial_cash=1000000, commission=0.0003, 
                 stock_code=None, plot=False):
    """
    执行回测
    
    Parameters:
    -----------
    df : DataFrame
        股票数据，必须包含 date, open, high, low, close, volume
    strategy_class : bt.Strategy
        策略类
    strategy_params : dict
        策略参数字典
    initial_cash : float
        初始资金
    commission : float
        手续费率
    stock_code : str
        指定股票代码（如果 df 包含多只股票）
    plot : bool
        是否绘图
    """
    
    # 筛选指定股票
    if stock_code:
        df = df[df['code'] == stock_code].copy()
    else:
        # 默认取第一只股票
        stock_code = df['code'].unique()[0]
        df = df[df['code'] == stock_code].copy()
    
    if len(df) == 0:
        print(f"❌ 股票 {stock_code} 无数据")
        return None
    
    # 准备数据
    df = df.set_index('date').sort_index()
    
    # 创建 Backtrader 数据源
    class PandasData(bt.feeds.PandasData):
        params = (
            ('datetime', None),
            ('open', 'open'),
            ('high', 'high'),
            ('low', 'low'),
            ('close', 'close'),
            ('volume', 'volume'),
            ('openinterest', -1),
        )
    
    # 初始化 Cerebro
    cerebro = bt.Cerebro()
    
    # 添加数据
    data = PandasData(dataname=df)
    cerebro.adddata(data)
    
    # 添加策略
    if strategy_params:
        cerebro.addstrategy(strategy_class, **strategy_params)
    else:
        cerebro.addstrategy(strategy_class)
    
    # 设置初始资金
    cerebro.broker.setcash(initial_cash)
    
    # 设置手续费
    cerebro.broker.setcommission(commission=commission)
    
    # 添加分析器
    cerebro.addanalyzer(bt.analyzers.SharpeRatio, _name='sharpe')
    cerebro.addanalyzer(bt.analyzers.DrawDown, _name='drawdown')
    cerebro.addanalyzer(bt.analyzers.TradeAnalyzer, _name='trades')
    cerebro.addanalyzer(bt.analyzers.Returns, _name='returns')
    
    # 打印回测前信息
    print(f"\n📊 回测配置:")
    print(f"   股票代码: {stock_code}")
    print(f"   起始资金: {initial_cash:,.0f}")
    print(f"   手续费率: {commission*100:.3f}%")
    print(f"   数据区间: {df.index.min().date()} ~ {df.index.max().date()} ({len(df)} 天)")
    
    # 运行回测
    print(f"\n🚀 开始回测...")
    results = cerebro.run()
    strat = results[0]
    
    # 打印结果
    final_value = cerebro.broker.getvalue()
    total_return = (final_value / initial_cash - 1) * 100
    
    print(f"\n📈 回测结果:")
    print(f"   最终资金: {final_value:,.0f}")
    print(f"   总收益率: {total_return:+.2f}%")
    
    # 分析器结果
    sharpe = strat.analyzers.sharpe.get_analysis()
    drawdown = strat.analyzers.drawdown.get_analysis()
    trades = strat.analyzers.trades.get_analysis()
    returns = strat.analyzers.returns.get_analysis()
    
    print(f"\n📊 风险指标:")
    if sharpe and sharpe.get('sharperatio'):
        print(f"   夏普比率: {sharpe['sharperatio']:.3f}")
    if drawdown:
        print(f"   最大回撤: {drawdown.get('drawdown', 0):.2f}%")
        print(f"   最大回撤资金: {drawdown.get('moneydown', 0):.0f}")
    
    if trades and trades.get('total'):
        total_trades = trades['total']['total']
        won_trades = trades['won']['total'] if trades.get('won') else 0
        lost_trades = trades['lost']['total'] if trades.get('lost') else 0
        win_rate = won_trades / total_trades * 100 if total_trades > 0 else 0
        
        print(f"\n📋 交易统计:")
        print(f"   总交易次数: {total_trades}")
        print(f"   盈利次数: {won_trades}")
        print(f"   亏损次数: {lost_trades}")
        print(f"   胜率: {win_rate:.1f}%")
    
    # 绘图
    if plot:
        cerebro.plot(style='candlestick', barup='red', bardown='green')
    
    return {
        'stock_code': stock_code,
        'final_value': final_value,
        'total_return': total_return,
        'sharpe': sharpe.get('sharperatio') if sharpe else None,
        'max_drawdown': drawdown.get('drawdown', 0) if drawdown else 0,
        'trades': trades
    }

print("✅ 回测函数定义完成")

### 5.2 在验证集上优化参数

In [ ]:
def grid_search_params(df, param_grid, stock_code=None):
    """
    网格搜索最优参数
    
    Parameters:
    -----------
    df : DataFrame
        验证集数据
    param_grid : dict
        参数网格，如 {'short_period': [5, 10], 'long_period': [20, 30]}
    """
    print(f"\n🔍 参数优化 (网格搜索)")
    print("="*60)
    
    results = []
    
    # 生成参数组合
    from itertools import product
    keys = list(param_grid.keys())
    values = list(param_grid.values())
    
    for combo in product(*values):
        params = dict(zip(keys, combo))
        
        # 确保短期 < 长期
        if params['short_period'] >= params['long_period']:
            continue
        
        print(f"\n测试参数: {params}")
        
        # 执行回测（关闭日志）
        result = run_backtest(
            df, DualMAStrategy, 
            strategy_params={**params, 'print_log': False},
            stock_code=stock_code,
            plot=False
        )
        
        if result:
            result.update(params)
            results.append(result)
            print(f"   收益率: {result['total_return']:+.2f}%, 夏普: {result.get('sharpe', 0) or 0:.3f}")
    
    # 排序找出最佳参数
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values('total_return', ascending=False)
    
    print(f"\n{'='*60}")
    print("📊 参数优化结果 (Top 5):")
    print(results_df[['short_period', 'long_period', 'total_return', 'sharpe']].head().to_string())
    
    # 返回最佳参数
    best = results_df.iloc[0]
    best_params = {
        'short_period': int(best['short_period']),
        'long_period': int(best['long_period']),
        'print_log': True
    }
    
    print(f"\n✅ 最优参数: {best_params}")
    return best_params, results_df

# 定义参数网格
param_grid = {
    'short_period': [5, 10, 15],
    'long_period': [20, 30, 60]
}

# 在验证集上优化（以茅台为例）
best_params, all_results = grid_search_params(val_df, param_grid, stock_code='600519')

### 5.3 在测试集上进行最终回测

In [ ]:
print("\n" + "="*60)
print("🎯 最终回测 (测试集 - 未见过数据)")
print("="*60)

# 使用最优参数在测试集上回测
final_result = run_backtest(
    test_df, 
    DualMAStrategy, 
    strategy_params=best_params,
    stock_code='600519',
    plot=True
)

### 5.4 多股票回测对比

In [ ]:
def backtest_multiple_stocks(df, stock_codes, strategy_class, strategy_params):
    """多股票回测"""
    results = []
    
    for code in stock_codes:
        print(f"\n{'='*60}")
        print(f"📊 回测股票: {code}")
        
        result = run_backtest(
            df, strategy_class, strategy_params,
            stock_code=code, plot=False
        )
        
        if result:
            results.append(result)
    
    # 汇总结果
    summary_df = pd.DataFrame(results)
    
    print(f"\n{'='*60}")
    print("📊 多股票回测汇总:")
    print("="*60)
    print(summary_df[['stock_code', 'total_return', 'sharpe', 'max_drawdown']].to_string())
    
    # 统计
    avg_return = summary_df['total_return'].mean()
    win_count = (summary_df['total_return'] > 0).sum()
    
    print(f"\n📈 统计:")
    print(f"   平均收益率: {avg_return:+.2f}%")
    print(f"   盈利股票数: {win_count}/{len(summary_df)}")
    print(f"   胜率: {win_count/len(summary_df)*100:.1f}%")
    
    return summary_df

# 获取所有测试集股票代码
test_stocks = test_df['code'].unique().tolist()

# 多股票回测
summary = backtest_multiple_stocks(
    test_df, 
    test_stocks, 
    DualMAStrategy, 
    best_params
)

### 5.5 买入持有基准对比

In [ ]:
def buy_and_hold_benchmark(df, stock_code, initial_cash=1000000):
    """计算买入持有基准收益"""
    stock_df = df[df['code'] == stock_code].copy().sort_values('date')
    
    if len(stock_df) == 0:
        return None
    
    first_price = stock_df['close'].iloc[0]
    last_price = stock_df['close'].iloc[-1]
    
    shares = int(initial_cash / first_price / 100) * 100
    final_value = shares * last_price
    total_return = (final_value / initial_cash - 1) * 100
    
    return {
        'strategy': 'Buy & Hold',
        'final_value': final_value,
        'total_return': total_return,
        'trades': 1
    }

# 对比策略 vs 买入持有
print("\n" + "="*60)
print("📊 策略对比 (测试集)")
print("="*60)

for code in test_stocks:
    # 策略收益
    strat_result = run_backtest(
        test_df, DualMAStrategy, best_params,
        stock_code=code, plot=False
    )
    
    # 买入持有收益
    bh_result = buy_and_hold_benchmark(test_df, code)
    
    if strat_result and bh_result:
        print(f"\n📈 {code}:")
        print(f"   双均线策略: {strat_result['total_return']:+.2f}%")
        print(f"   买入持有:   {bh_result['total_return']:+.2f}%")
        print(f"   超额收益:   {strat_result['total_return'] - bh_result['total_return']:+.2f}%")

## 6. 完整流程总结

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║                   双均线策略回测总结                          ║
╠══════════════════════════════════════════════════════════════╣
║ 1. 数据加载                                                   ║
║    - 从 API/CSV 加载股票数据                                  ║
║    - 添加市值数据                                            ║
║                                                               ║
║ 2. 数据筛选                                                   ║
║    - 主板股票 (SH/SZ)                                        ║
║    - 市值 >= 100亿                                           ║
║                                                               ║
║ 3. 时间序列划分 (严格无穿越)                                   ║
║    - 训练集: 60% (参数学习)                                   ║
║    - 验证集: 20% (参数优化)                                   ║
║    - 测试集: 20% (最终评估)                                   ║
║                                                               ║
║ 4. 策略回测                                                   ║
║    - 金叉买入, 死叉卖出                                       ║
║    - 验证集网格搜索最优参数                                    ║
║    - 测试集评估策略表现                                       ║
║    - 与买入持有基准对比                                       ║
╚══════════════════════════════════════════════════════════════╝
""")

# 显示关键参数
print(f"\n📋 最优参数配置:")
print(f"   短期均线周期: {best_params['short_period']}日")
print(f"   长期均线周期: {best_params['long_period']}日")
print(f"\n📊 数据划分:")
print(f"   训练集: {len(train_df):,} 条")
print(f"   验证集: {len(val_df):,} 条")
print(f"   测试集: {len(test_df):,} 条")

## 附录: 保存 Notebook 状态

保存当前 Notebook 和数据以便后续使用

In [ ]:
# 保存数据集
output_dir = "data/processed"
os.makedirs(output_dir, exist_ok=True)

# 保存各数据集
train_df.to_csv(f"{output_dir}/train_set.csv", index=False)
val_df.to_csv(f"{output_dir}/val_set.csv", index=False)
test_df.to_csv(f"{output_dir}/test_set.csv", index=False)

# 保存最优参数
import json
with open(f"{output_dir}/best_params.json", 'w') as f:
    json.dump(best_params, f, indent=2)

print(f"✅ 数据已保存到 {output_dir}/")
print(f"   - train_set.csv ({len(train_df):,} 条)")
print(f"   - val_set.csv ({len(val_df):,} 条)")
print(f"   - test_set.csv ({len(test_df):,} 条)")
print(f"   - best_params.json")